In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

In [2]:
product = torch.mul
mins = lambda a, b: torch.minimum(a, b)
maxs = lambda a, b: torch.maximum(a, b)
mean = lambda a, b: (a + b) / 2.0
gmean = lambda a, b: torch.sqrt(torch.clamp(a, min=0) * torch.clamp(b, min=0))

def softmin(a, b, k=10.0):
    stacked_tensors = torch.stack([a, b], dim=-1)
    weights = F.softmax(-k * stacked_tensors, dim=-1)
    return torch.sum(weights * stacked_tensors, dim=-1)

In [3]:
import torch
import torch.nn as nn
from typing import Tuple

@torch.jit.script
def fused_tversky_score(a_fk: torch.Tensor,
                        b_fk: torch.Tensor,
                        constants: torch.Tensor,
                        diff_mode_sub: bool) -> torch.Tensor:

    mask_intersect = (a_fk > 0) & (b_fk > 0)
    intersection = torch.where(mask_intersect, a_fk * b_fk, 0.0)

    if diff_mode_sub:
        mask_ab = mask_intersect & (a_fk > b_fk)
        diff_ab = torch.where(mask_ab, a_fk - b_fk, 0.0)

        mask_ba = mask_intersect & (b_fk > a_fk)
        diff_ba = torch.where(mask_ba, b_fk - a_fk, 0.0)
    else:
        mask_ab = (a_fk > 0) & (b_fk <= 0)
        diff_ab = torch.where(mask_ab, a_fk, 0.0)

        mask_ba = (b_fk > 0) & (a_fk <= 0)
        diff_ba = torch.where(mask_ba, b_fk, 0.0)

    weighted_elements = (intersection * constants[0] +
                         diff_ab * constants[1] +
                         diff_ba * constants[2])

    return torch.sum(weighted_elements, dim=-1)


class TverskySimilarity(nn.Module):
    def __init__(self, feats: torch.Tensor, intersection_mode: str = "product", difference_mode: str = "ignorematch"):
        super().__init__()
        self.feats = feats
        self.intersection_mode = intersection_mode
        self.difference_mode = difference_mode
        self.use_sub_match = (difference_mode == "substractmatch")

        self.constants = nn.Parameter(torch.tensor([1.0, -1.0, -1.0]), requires_grad=True)

    def forward(self, a: torch.Tensor, b_batch: torch.Tensor) -> torch.Tensor:
        """
        a: (B, T, D) - RNN Input
        b_batch: (B, T, P, D) - Dynamic Projections
        """
        feats_T = self.feats.t()  # (D, K)

        # (B, T, 1, K)
        if a.dim() == 3:
            B, T, D = a.shape
            a_flat = a.reshape(B * T, D)
            a_fk = torch.matmul(a_flat, feats_T).reshape(B, T, 1, -1)
        else:
             a_fk = torch.matmul(a, feats_T).unsqueeze(1)

        # (B, T, P, K)
        if b_batch.dim() == 4:
            B, T, P, D = b_batch.shape
            b_flat = b_batch.reshape(B * T * P, D)
            b_fk = torch.matmul(b_flat, feats_T).reshape(B, T, P, -1)
        else:
             b_fk = torch.matmul(b_batch, feats_T)

        result = fused_tversky_score(a_fk, b_fk, self.constants, self.use_sub_match)

        return result

In [4]:
import sys
sys.path.append("vision-lstm/vision_lstm")

In [5]:
from vision_lstm2 import *

In [6]:
class TverskyMatrixLSTMCell(MatrixLSTMCell):
    def __init__(self, dim, num_heads, norm_bias=True,
                 num_tversky_features=32, tversky_output_dim=None,
                 gate_rank=None):
        super().__init__(dim, num_heads, norm_bias)

        self.tversky_dim = dim
        output_dim = tversky_output_dim if tversky_output_dim else dim

        self.features = nn.Parameter(torch.randn(num_tversky_features, self.tversky_dim))
        self.base_projections = nn.Parameter(torch.randn(output_dim, self.tversky_dim))
        self.input_gate_proj = nn.Linear(self.tversky_dim, self.tversky_dim)

        self.proto_gate_param = nn.Parameter(torch.randn(num_tversky_features, self.tversky_dim))

        self.similarity_calculator = TverskySimilarity(
            feats=self.features,
            difference_mode="ignorematch"
        )

        self.output_proj = nn.Linear(output_dim, self.tversky_dim)
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.features, std=0.02)
        nn.init.normal_(self.base_projections, std=0.02)
        nn.init.xavier_uniform_(self.input_gate_proj.weight)
        nn.init.constant_(self.input_gate_proj.bias, 1.0)
        nn.init.normal_(self.proto_gate_param, std=0.02)

    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
        h_state = super().forward(q, k, v)

        if v.ndim == 4:
            B, NH, S, DH = v.shape
            v_seq = v.transpose(1, 2).reshape(B, S, -1)
        else:
            v_seq = v
            B, S, D = v_seq.shape

        input_gate = torch.sigmoid(self.input_gate_proj(h_state))
        v_gated = v_seq * input_gate

        proto_gate = torch.sigmoid(self.proto_gate_param)

        projections_gated = self.base_projections * proto_gate

        projections_ready = projections_gated.unsqueeze(0).unsqueeze(0)
        tversky_score = self.similarity_calculator(v_gated, projections_ready)

        tversky_out = self.output_proj(tversky_score)

        return h_state + tversky_out

In [7]:
def convert_vil_to_tversky(model, num_tversky_features=32):
    count = 0
    for name, module in model.named_modules():
        if hasattr(module, 'mlstm_cell') and isinstance(module.mlstm_cell, MatrixLSTMCell):
            old_cell = module.mlstm_cell
            dim = old_cell.dim
            num_heads = old_cell.num_heads
            norm_bias = (old_cell.outnorm.bias is not None)

            new_cell = TverskyMatrixLSTMCell(
                dim=dim,
                num_heads=num_heads,
                norm_bias=norm_bias,
                num_tversky_features=num_tversky_features,
                tversky_output_dim=64,
                gate_rank=64
            )

            new_cell.to(next(old_cell.parameters()).device)
            new_cell.to(next(old_cell.parameters()).dtype)

            module.mlstm_cell = new_cell
            count += 1

    print(f"Successfully upgraded {count} mLSTM cells to Tversky-mLSTM cells.")
    return model

In [8]:
import torch
import torch.nn as nn
import torchvision.models as models


class StandardViL(nn.Module):
    def __init__(self, vil_config):
        super().__init__()

        print("Initializing ResNet18 Backbone...")
        weights = models.ResNet18_Weights.IMAGENET1K_V1
        resnet = models.resnet18(weights=weights)
        self.backbone = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
            resnet.layer1, resnet.layer2, resnet.layer3
        )
        for param in self.backbone.parameters():
            param.requires_grad = False

        resnet_dim = 256
        target_dim = vil_config['dim']

        if resnet_dim != target_dim:
            self.proj = nn.Linear(resnet_dim, target_dim)
        else:
            self.proj = nn.Identity()

        self.vil = VisionLSTM2(**vil_config)

        self.vil.pos_embed.embed = nn.Parameter(torch.zeros(1, 196, target_dim))
        torch.nn.init.trunc_normal_(self.vil.pos_embed.embed, std=0.02)
    
    def train(self, mode=True):
            super().train(mode)
            self.backbone.eval()

    def forward(self, x):
        with torch.no_grad():
            x = self.backbone(x)
        x = x.flatten(2).transpose(1, 2)
        x = self.proj(x)

        x = x + self.vil.pos_embed.embed

        for block in self.vil.blocks:
            x = block(x)
        x = self.vil.norm(x)
        x_out = torch.cat([x[:, 0], x[:, -1]], dim=1)
        return x_out

In [9]:
class WeaklySupervisedTripletLoss(nn.Module):
    def __init__(self, margin=0.1):
        super().__init__()
        self.margin = margin

    def forward(self, q, pos_set, neg_set):
        pos_dist_sq = (q.unsqueeze(1) - pos_set).pow(2).sum(dim=-1)
        best_pos_sq, _ = pos_dist_sq.min(dim=1)
        neg_dist_sq = (q.unsqueeze(1) - neg_set).pow(2).sum(dim=-1)

        loss_per_neg = F.relu(best_pos_sq.unsqueeze(1) + self.margin - neg_dist_sq)
        return loss_per_neg.sum(dim=1).mean()

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from datasets import load_dataset
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import gc
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# Standard ViL (Depth 12)
conf_std = {
    "dim": 256,
    "depth": 12,
    "patch_size": 16,
    "input_shape": (3, 224, 224),
    "mode": "features",
    "pooling": "bilateral_flatten",
    "drop_path_rate": 0.0
}

# Tversky ViL (Depth 6)
conf_tvy = {
    "dim": 256,
    "depth": 6,
    "patch_size": 16,
    "input_shape": (3, 224, 224),
    "mode": "features",
    "pooling": "bilateral_flatten",
    "drop_path_rate": 0.0
}

m_std = VisionLSTM2(**conf_std)


m_tvy = VisionLSTM2(**conf_tvy)
m_tvy = convert_vil_to_tversky(m_tvy, num_tversky_features=64)

params_std = count_params(m_std)
params_tvy = count_params(m_tvy)

print(f"Standard ViL Params: {params_std:,}")
print(f"Tversky ViL Params:  {params_tvy:,}")
print(f"Difference:          +{params_tvy - params_std:,} parameters")
del m_tvy
del m_std
gc.collect()
torch.cuda.empty_cache()

/home/ned/Tverksy/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Successfully upgraded 12 mLSTM cells to Tversky-mLSTM cells.
Standard ViL Params: 10,366,912
Tversky ViL Params:  10,038,148
Difference:          +-328,764 parameters


/home/ned/Tverksy/vision-lstm/vision_lstm/vision_lstm2.py:698: UserWarning: passed mode=features -> output_shape is ignored ((1000,))
  warnings.warn(f"passed mode=features -> output_shape is ignored ({self.output_shape})")


In [11]:

import random

def set_seed(seed_value=42):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True
    print(f"Seed set to {seed_value}")

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os
import glob
import random
import numpy as np
from tqdm import tqdm


MODEL_NAME = "Standard_ViL"
DATA_ROOT = "Nordland"
REF_SEASON = "summer"
QUERY_SEASON = "winter"
BATCH_SIZE = 2
NUM_POSITIVES = 2
NUM_NEGATIVES = 6
POOL_SIZE = 500
LR = 1e-4
MAX_EPOCHS = 50
PATIENCE = 5
CACHE_REFRESH_FREQ_INIT = 2
WARMUP_EPOCHS = 4
MARGIN = 0.1 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# TRAIN_RANGE = range(0, 200)
# VAL_RANGE   = range(15000, 15100)
# TEST_RANGE  = range(14, 15)

TRAIN_RANGE = range(0, 15000)
VAL_RANGE   = range(15000, 17000)
TEST_RANGE  = range(18000, 22000)



class NordlandNetVLADDataset(Dataset):
    """
    Returns (anchor, pos_stack, neg_stack) tuples.

    Hard negative mining uses:
      - A random pool of POOL_SIZE candidates (paper: 1000)
      - The K hardest from that pool (current epoch cache)
      - The hard negatives remembered from the PREVIOUS epoch
        (paper: "remembering previous hard negatives adds stability")
    """
    def __init__(self, root, ref_season, query_season, indices, transform=None):
        self.indices   = list(indices)
        self.transform = transform
        self.ref_images   = sorted(glob.glob(os.path.join(root, ref_season,   "*.png")))
        self.query_images = sorted(glob.glob(os.path.join(root, query_season, "*.png")))

        # Set externally after each cache build
        self.feature_cache = None

        # Stores {real_idx: [neg_real_indices]} populated at end of each epoch
        self.prev_hard_neg_indices: dict = {}

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]

        # ── 1. ANCHOR ──────────────────────────────────────────────────────────
        anchor_img = self._load(self.query_images[real_idx])

        # ── 2. POTENTIAL POSITIVES (~10 metres → adjacent frames) ─────────────
        pos_imgs = []
        for offset in range(NUM_POSITIVES):
            p_idx = min(max(real_idx + offset, 0), len(self.ref_images) - 1)
            pos_imgs.append(self._load(self.ref_images[p_idx]))
        pos_imgs = torch.stack(pos_imgs)

        # ── 3. HARD NEGATIVES ─────────────────────────────────────────────────
        neg_imgs = []

        if self.feature_cache is not None:
            anchor_feat = self.feature_cache[real_idx]

            # Valid negatives: further than 25 metres (≈ 25 frame indices)
            valid_indices = [i for i in self.indices if abs(i - real_idx) > 25]

            # Current-epoch pool: POOL_SIZE random samples
            pool_now = random.sample(valid_indices, min(POOL_SIZE, len(valid_indices)))

            # Previous-epoch hard negatives (may overlap with pool_now — that's fine)
            pool_prev = self.prev_hard_neg_indices.get(real_idx, [])

            # Merge and deduplicate
            combined_pool = list(set(pool_now + pool_prev))

            # Score by Squared L2 in feature space
            pool_feats = self.feature_cache[combined_pool]
            distances  = (pool_feats - anchor_feat).pow(2).sum(dim=-1)

            # Take K hardest (smallest distance = hardest)
            k = min(NUM_NEGATIVES, len(combined_pool))
            _, hard_rel = torch.topk(distances, k, largest=False)
            chosen_neg_indices = [combined_pool[r.item()] for r in hard_rel]

            # Save for next epoch (stability, per paper)
            self.prev_hard_neg_indices[real_idx] = chosen_neg_indices

            for ni in chosen_neg_indices:
                neg_imgs.append(self._load(self.ref_images[ni]))

        else:
            # First epoch: random negatives (no cache yet)
            while len(neg_imgs) < NUM_NEGATIVES:
                n_idx = random.choice(self.indices)
                if abs(n_idx - real_idx) > 25:
                    neg_imgs.append(self._load(self.ref_images[n_idx]))

        return anchor_img, pos_imgs, torch.stack(neg_imgs)

    def _load(self, path):
        img = Image.open(path).convert("RGB")
        return self.transform(img) if self.transform else img


class SimpleCacheDataset(Dataset):
    """Minimal dataset for fast reference-set feature extraction."""
    def __init__(self, root, season, indices, transform):
        self.images  = sorted(glob.glob(os.path.join(root, season, "*.png")))
        self.indices = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        img = self.transform(Image.open(self.images[real_idx]).convert("RGB"))
        return img, real_idx


class NordlandEvalDataset(Dataset):
    def __init__(self, root, season, indices, transform):
        all_imgs = sorted(glob.glob(os.path.join(root, season, "*.png")))
        self.images    = [all_imgs[i] for i in indices]
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.transform(Image.open(self.images[idx]).convert("RGB"))


def get_transforms():
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])


def evaluate_on_loaders(model, ref_loader, query_loader, device):
    model.eval()
    refs, queries = [], []
    with torch.no_grad():
            for img in ref_loader:
                refs.append(model(img.to(device)).cpu())
            for img in query_loader:
                queries.append(model(img.to(device)).cpu())

    refs    = F.normalize(torch.cat(refs),    p=2, dim=1, eps=1e-5)
    queries = F.normalize(torch.cat(queries), p=2, dim=1, eps=1e-5)

    sim_matrix = torch.mm(queries, refs.t())

    gt_indices = torch.arange(len(queries)).unsqueeze(1)

    _, top10 = torch.topk(sim_matrix, k=10, dim=1)

    matches = (top10.cpu() == gt_indices)

    r1  = matches[:, :1].any(dim=1).float().mean().item() * 100
    r5  = matches[:, :5].any(dim=1).float().mean().item() * 100
    r10 = matches[:, :10].any(dim=1).float().mean().item() * 100

    return r1, r5, r10


def update_feature_cache(model, transform, device, total_db_size):
    """
    Builds a CPU-side float32 feature cache of the reference (DB) set.
    Used for hard-negative mining without holding GPU graphs.

    Memory note:
      cache shape = (15001, 512) float32 ≈ 30 MB on CPU — negligible.
      We use a larger batch here (32) because no gradients are tracked.
    """
    print("\n>>> Building Feature Cache for Hard Negative Mining...")
    model.eval()

    cache_ds     = SimpleCacheDataset(DATA_ROOT, REF_SEASON, TRAIN_RANGE, transform)
    cache_loader = DataLoader(cache_ds, batch_size=64, num_workers=2,
                          pin_memory=True, shuffle=False)

    embed_dim = model.vil.dim * 2
    cache     = torch.zeros((total_db_size, embed_dim), device='cpu')

    with torch.no_grad():
        for imgs, indices in tqdm(cache_loader, desc="  Caching DB"):
            feats          = model(imgs.to(device))
            feats          = F.normalize(feats, p=2, dim=1, eps=1e-5)
            cache[indices] = feats.cpu()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return cache


def train_hybrid_model():
    print(f"\n=== Starting Run: {MODEL_NAME} ===")

    gc.collect()
    if torch.cuda.is_available():
        print("[INFO] Clearing GPU cache before training...")
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    
    set_seed(42)

    # Note: Assuming conf_std is defined globally in your script based on your snippet!
    model     = StandardViL(vil_config=conf_std).to(DEVICE)
    # print(model.vil.blocks[0].rowwise_from_top_left.layer.mlstm_cell.similarity_calculator.constants)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    criterion = WeaklySupervisedTripletLoss(margin=MARGIN)

    # NetVLAD Paper: halve LR every 5 epochs
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    # 1. State Variables & Paths
    start_epoch        = 0
    best_val_recall    = 0.0
    patience_counter   = 0
    cache_refresh_freq = CACHE_REFRESH_FREQ_INIT
    max_idx            = max(TRAIN_RANGE) + 1

    # --- NEW: Temporary variables to hold saved cache ---
    saved_feature_cache = None
    saved_prev_hard_neg = {}

    
    checkpoint_path = f"{MODEL_NAME}_latest_checkpoint.pth"
    best_model_path = f"{MODEL_NAME}_best.pth"

    # 2. Checkpoint Resume Logic
    if os.path.exists(checkpoint_path):
        print(f"\n[INFO] Found checkpoint '{checkpoint_path}'. Resuming training...")
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE)

        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

        start_epoch        = checkpoint['epoch'] + 1
        best_val_recall    = checkpoint['best_val_recall']
        cache_refresh_freq = checkpoint['cache_refresh_freq']
        patience_counter   = checkpoint.get('patience_counter', 0)
        saved_feature_cache = checkpoint.get('feature_cache', None)

        # --- NEW: Extract dataset memory ---
        if saved_feature_cache is not None:
            saved_feature_cache = saved_feature_cache.cpu()
        saved_prev_hard_neg = checkpoint.get('prev_hard_neg_indices', {})

        print(f"[INFO] Resuming from Epoch {start_epoch+1}")
    else:
        print("\n[INFO] No checkpoint found. Starting from scratch.")

    # 3. DataLoaders
    transform    = get_transforms()
    train_ds     = NordlandNetVLADDataset(DATA_ROOT, REF_SEASON, QUERY_SEASON, TRAIN_RANGE, transform)

    # --- NEW: Inject the saved cache back into the dataset ---
    if saved_feature_cache is not None:
        print("[INFO] Restored Feature Cache and Hard Negative Memory from checkpoint.")
        train_ds.feature_cache = saved_feature_cache
        train_ds.prev_hard_neg_indices = saved_prev_hard_neg



    train_loader = DataLoader(
      train_ds, batch_size=BATCH_SIZE, shuffle=True,
      num_workers=2, pin_memory=True,
      prefetch_factor=2,
  )

    val_ref_loader = DataLoader(
        NordlandEvalDataset(DATA_ROOT, REF_SEASON,   VAL_RANGE, transform),
        batch_size=16, num_workers=2, pin_memory=True
    )
    val_q_loader = DataLoader(
        NordlandEvalDataset(DATA_ROOT, QUERY_SEASON, VAL_RANGE, transform),
        batch_size=16, num_workers=2, pin_memory=True
    )

    # 4. Main Training Loop (Starts at start_epoch)
    for epoch in range(start_epoch, MAX_EPOCHS):

   # ← add these
        # Only build cache IF we are past the warmup phase
        if epoch >= WARMUP_EPOCHS and (epoch - WARMUP_EPOCHS) % cache_refresh_freq == 0:
            feature_cache = update_feature_cache(model, transform, DEVICE, max_idx)
            train_loader.dataset.feature_cache = feature_cache

        # ── B. Train Epoch ────────────────────────────────────────────────────
        model.train()
        total_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{MAX_EPOCHS}")

        for anchor, pos_set, neg_set in pbar:
            # Get shapes to reshape sets dynamically
            B, num_pos, C, H, W = pos_set.shape
            _, num_neg, _, _, _ = neg_set.shape

            # Move to GPU
            anchor   = anchor.to(DEVICE)
            pos_flat = pos_set.view(-1, C, H, W).to(DEVICE)
            neg_flat = neg_set.view(-1, C, H, W).to(DEVICE)

            optimizer.zero_grad()
            a_out = F.normalize(model(anchor),   p=2, dim=-1, eps=1e-5)
            p_out = F.normalize(model(pos_flat), p=2, dim=-1, eps=1e-5).view(B, num_pos, -1)
            n_out = F.normalize(model(neg_flat), p=2, dim=-1, eps=1e-5).view(B, num_neg, -1)
            loss = criterion(a_out, p_out, n_out)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()



            total_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}",
                              "lr":   f"{scheduler.get_last_lr()[0]:.2e}"})

        # ── C. LR Step + Adaptive Cache Refresh (paper §training) ─────────────
        scheduler.step()

        # Every 5 epochs the LR halves → also halve recomputation frequency
        if (epoch + 1) % 5 == 0:
            print(f"  [Scheduler] LR halved to {scheduler.get_last_lr()[0]:.2e}.")

        # ── D. Validate & Save ────────────────────────────────────────────────
        val_r1, val_r5, val_r10 = evaluate_on_loaders(model, val_ref_loader, val_q_loader, DEVICE)
        avg_loss = total_loss / len(train_loader)



        print(f"Epoch {epoch+1:02d} | Loss={avg_loss:.4f} | LR={scheduler.get_last_lr()[0]:.2e}")
        print(f"  > R@1: {val_r1:.2f}% | R@5: {val_r5:.2f}% | R@10: {val_r10:.2f}%")

        # --- Early Stopping & Best Model Logic ---
        # Note: We still use R@1 as the benchmark for "Best Model", as is standard in VPR
        if val_r1 > best_val_recall:
            best_val_recall = val_r1
            patience_counter = 0
            torch.save(model.state_dict(), best_model_path)
            print(f"  >>> Best Model Saved! (R@1: {best_val_recall:.2f}%)")
        else:
            patience_counter += 1
            print(f"  ... No improvement. Patience: {patience_counter}/{PATIENCE}")

        # --- Save Full Checkpoint Every Epoch ---
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_recall': best_val_recall,
            'patience_counter': patience_counter,
            'cache_refresh_freq': cache_refresh_freq,
            'feature_cache': train_loader.dataset.feature_cache,
            'prev_hard_neg_indices': train_loader.dataset.prev_hard_neg_indices,
        }
        torch.save(checkpoint, checkpoint_path)

        # Free any lingering GPU memory between epochs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # --- Trigger Early Stopping ---
        if patience_counter >= PATIENCE:
            print(f"\n[Early Stopping] Triggered at Epoch {epoch+1}. No improvement for {PATIENCE} epochs.")
            break

    # 5. Final Test Evaluation
    # 5. Final Test Evaluation
    print(f"\nLoading Best Model for Final Test Set Evaluation...")
    model.load_state_dict(torch.load(best_model_path))

    test_ref_loader = DataLoader(NordlandEvalDataset(DATA_ROOT, REF_SEASON, TEST_RANGE, transform), batch_size=16)
    test_q_loader   = DataLoader(NordlandEvalDataset(DATA_ROOT, QUERY_SEASON, TEST_RANGE, transform), batch_size=16)

    final_r1, final_r5, final_r10 = evaluate_on_loaders(model, test_ref_loader, test_q_loader, DEVICE)
    print(f"\nFinal Test Scores for {MODEL_NAME}:")
    print(f"R@1:  {final_r1:.2f}%")
    print(f"R@5:  {final_r5:.2f}%")
    print(f"R@10: {final_r10:.2f}%")

    return final_r1


In [ ]:
train_hybrid_model()


=== Starting Run: Standard_ViL ===
[INFO] Clearing GPU cache before training...
Seed set to 42
Initializing ResNet18 Backbone...


/home/ned/Tverksy/vision-lstm/vision_lstm/vision_lstm2.py:698: UserWarning: passed mode=features -> output_shape is ignored ((1000,))
  warnings.warn(f"passed mode=features -> output_shape is ignored ({self.output_shape})")



[INFO] No checkpoint found. Starting from scratch.


Epoch 1/50: 100%|██████████| 7500/7500 [4:04:10<00:00,  1.95s/it, loss=0.0000, lr=1.00e-04]  


Epoch 01 | Loss=0.1667 | LR=1.00e-04
  > R@1: 11.55% | R@5: 28.80% | R@10: 40.30%
  >>> Best Model Saved! (R@1: 11.55%)


...

Epoch 8/50: 100%|██████████| 7500/7500 [4:04:42<00:00,  1.96s/it, loss=0.0000, lr=5.00e-05]  


Epoch 08 | Loss=0.0112 | LR=5.00e-05
  > R@1: 57.15% | R@5: 83.25% | R@10: 89.90%
  >>> Best Model Saved! (R@1: 57.15%)

>>> Building Feature Cache for Hard Negative Mining...


Epoch 9/50: 100%|██████████| 7500/7500 [4:04:37<00:00,  1.96s/it, loss=0.0000, lr=5.00e-05]  


Epoch 09 | Loss=0.0108 | LR=5.00e-05
  > R@1: 57.50% | R@5: 85.40% | R@10: 91.25%
  >>> Best Model Saved! (R@1: 57.50%)


Epoch 10/50:  48%|████▊     | 3590/7500 [1:57:05<2:07:32,  1.96s/it, loss=0.0000, lr=5.00e-05]

In [ ]:
train_hybrid_model()


=== Starting Run: Standard_ViL ===
[INFO] Clearing GPU cache before training...
Seed set to 42
Initializing ResNet18 Backbone...

[INFO] Found checkpoint 'Standard_ViL_latest_checkpoint.pth'. Resuming training...
[INFO] Resuming from Epoch 10
[INFO] Restored Feature Cache and Hard Negative Memory from checkpoint.


Epoch 10/50: 100%|██████████| 7500/7500 [4:02:09<00:00,  1.94s/it, loss=nan, lr=5.00e-05]  

  [Scheduler] LR halved. Cache refresh freq → every 8 epochs.


Epoch 10 | Loss=nan | LR=2.50e-05
  > R@1: 0.05% | R@5: 0.25% | R@10: 0.50%
  ... No improvement. Patience: 1/5


Epoch 11/50:   8%|▊         | 624/7500 [19:23<3:33:41,  1.86s/it, loss=nan, lr=2.50e-05]


KeyboardInterrupt: 

# After adding gradient clipping

`tversky` didn't need this!

Had to restart as the checkpoint was messed up due to this explosion, (and also bcuz of the power cut :) )

In [13]:
train_hybrid_model()


=== Starting Run: Standard_ViL ===
[INFO] Clearing GPU cache before training...
Seed set to 42
Initializing ResNet18 Backbone...

[INFO] No checkpoint found. Starting from scratch.


Epoch 1/50: 100%|██████████| 7500/7500 [1:55:45<00:00,  1.08it/s, loss=0.0000, lr=1.00e-04]


Epoch 01 | Loss=0.0337 | LR=1.00e-04
  > R@1: 14.75% | R@5: 33.75% | R@10: 44.00%
  >>> Best Model Saved! (R@1: 14.75%)


Epoch 2/50: 100%|██████████| 7500/7500 [1:55:54<00:00,  1.08it/s, loss=0.0000, lr=1.00e-04]


Epoch 02 | Loss=0.0082 | LR=1.00e-04
  > R@1: 24.30% | R@5: 52.90% | R@10: 65.75%
  >>> Best Model Saved! (R@1: 24.30%)


...

Epoch 24 | Loss=0.0053 | LR=6.25e-06
  > R@1: 80.35% | R@5: 95.75% | R@10: 98.45%
  ... No improvement. Patience: 4/5

>>> Building Feature Cache for Hard Negative Mining...


Epoch 25/50: 100%|██████████| 7500/7500 [2:11:07<00:00,  1.05s/it, loss=0.0000, lr=6.25e-06]  

  [Scheduler] LR halved to 3.13e-06.


Epoch 25 | Loss=0.0052 | LR=3.13e-06
  > R@1: 79.65% | R@5: 96.00% | R@10: 98.50%
  ... No improvement. Patience: 5/5

[Early Stopping] Triggered at Epoch 25. No improvement for 5 epochs.

Loading Best Model for Final Test Set Evaluation...

Final Test Scores for Standard_ViL:
R@1:  47.08%
R@5:  71.75%
R@10: 77.78%


47.075000405311584